# TimeBraid inference task examples

This notebook demonstrates the public single-request inference interface for TimeBraid. It uses synthetic time series created for this example; it does not load private datasets or evaluation code.

The examples exercise three structural routes:

| Route | `timeseries` | `horizon` | Canonical result |
| --- | --- | --- | --- |
| Text-only completion | omitted | `None` | generated `content` |
| Time-series understanding | one or more finite series | `None` | TS-conditioned free-text `content` |
| Point forecasting | one or more finite histories; select the target when multiple | positive integer | `content` plus exactly `horizon` numerical values for one target |

Description, question answering, anomaly reasoning, and multi-series comparison are prompt-level tasks on the same understanding route; they are not separate structured output schemas. Forecasting returns one target series. With multiple input series, pass `target_series_index` explicitly to the processor; a single input defaults to index `0`. The forecast display cards below use one input history.


## Runtime configuration

Install TimeBraid, the CUDA-matched Torch build, and FlashAttention as described in the repository README. The plotted examples additionally require the notebook extra:

```bash
python -m pip install '.[notebook]'
```

The notebook defaults to `XinyueWangg/TimeBraid-2.5B` and downloads it on first use. To use another complete local checkpoint or Hub ID, set `TIMEBRAID_MODEL` before starting the kernel; an existing setting is preserved. Select the physical GPU with `CUDA_VISIBLE_DEVICES` before kernel startup; the notebook uses the visible device as logical `cuda:0`.


In [ ]:
import os
import textwrap
from collections.abc import Sequence

import matplotlib.pyplot as plt
import torch
from matplotlib.ticker import MaxNLocator
from transformers import AutoModelForCausalLM, AutoProcessor

from timebraid import TimeBraidProcessor  # noqa: F401 - registers HF AutoClasses

In [ ]:
# Preserve a checkpoint selected by the user before starting the kernel.
os.environ.setdefault("TIMEBRAID_MODEL", "XinyueWangg/TimeBraid-2.5B")

In [ ]:
MODEL_ID_OR_PATH = os.environ.get("TIMEBRAID_MODEL")
if not MODEL_ID_OR_PATH:
    raise ValueError("Set TIMEBRAID_MODEL to a complete TimeBraid artifact or Hub ID.")
if not torch.cuda.is_available():
    raise RuntimeError("TimeBraid mixed time-series inference requires CUDA.")

import flash_attn  # noqa: E402, F401 - fail before loading model weights

processor = AutoProcessor.from_pretrained(
    MODEL_ID_OR_PATH,
    fix_mistral_regex=False,
    trust_remote_code=False,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_OR_PATH,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map={"": "cuda:0"},
    trust_remote_code=False,
    use_safetensors=True,
).eval()

## Shared request runner

Pass raw-scale values through `timeseries`; do not write `<stats>`, `<ts>`, `</ts>`, or reasoning-mode markers in the prompt. `TimeBraidProcessor` owns normalization and prompt construction. The same returned batch feature must be passed to `post_process_generation` so numerical forecasts can be validated and restored to the history scale.

Each call renders the exact request beside that run's model response. Generated text is a sample model output, not a gold label. Input series stay on their original scales; forecasts use the denormalized point values returned by the processor. No confidence bands, anomaly scores, or causal effects are inferred by the plotting layer.


`run_contrast_requests` submits the same history under several narratives and draws every outcome in one figure, so counterfactual pairs can be compared directly.

In [ ]:
_INPUT_COLOR = "#0072B2"
_OUTPUT_COLOR = "#D55E00"
_BOUNDARY_COLOR = "#555555"
_SERIES_COLORS = ("#0072B2", "#009E73", "#CC79A7", "#E69F00")


def _wrap_plot_text(text: str, *, width: int = 58) -> str:
    paragraphs = text.splitlines() or [""]
    return "\n".join(
        textwrap.fill(paragraph.strip(), width=width) if paragraph.strip() else ""
        for paragraph in paragraphs
    )


def _draw_text_panel(ax, *, heading: str, body: str, accent: str) -> None:
    ax.set_facecolor("#F7F8FA")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color("#D9DCE1")
        spine.set_linewidth(0.9)
    ax.text(
        0.04,
        0.94,
        heading,
        transform=ax.transAxes,
        va="top",
        color=accent,
        fontsize=12,
        fontweight="bold",
        parse_math=False,
    )
    ax.text(
        0.04,
        0.84,
        body,
        transform=ax.transAxes,
        va="top",
        color="#202124",
        fontsize=10,
        linespacing=1.35,
        parse_math=False,
    )


def _style_series_axis(ax, *, forecast: bool) -> None:
    ax.set_xlabel("Observation / forecast index" if forecast else "Observation index")
    ax.set_ylabel("Value (raw scale)")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", color="#D9DCE1", linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def render_request_result(
    *,
    title: str,
    messages: tuple[tuple[str, str], ...],
    series: tuple[tuple[float, ...], ...],
    horizon: int | None,
    result: dict,
    series_names: Sequence[str] | None = None,
):
    names = (
        tuple(f"Series {index + 1}" for index in range(len(series)))
        if series_names is None
        else tuple(series_names)
    )
    if len(names) != len(series):
        raise ValueError(
            f"series_names must align with series: {len(names)} != {len(series)}."
        )

    if horizon is None:
        if result["timeseries"] is not None:
            raise ValueError("A horizon-free request returned numerical output.")
        forecast_values: tuple[float, ...] = ()
        normalized_values: tuple[float, ...] = ()
    else:
        if len(series) != 1 or result["timeseries"] is None:
            raise ValueError(
                "This notebook forecast card expects one selected history and numerical output."
            )
        ts_output = result["timeseries"]
        forecast_values = tuple(ts_output["values"])
        normalized_values = tuple(ts_output["normalized_values"])
        if len(forecast_values) != horizon or len(normalized_values) != horizon:
            raise ValueError(
                "Forecast output lengths must equal "
                f"horizon={horizon}, got {len(forecast_values)} and "
                f"{len(normalized_values)}."
            )

    if not series:
        route = "text-only completion"
        series_summary = "timeseries=None"
    elif horizon is None:
        route = "time-series understanding"
        series_summary = f"timeseries lengths={[len(values) for values in series]}"
    else:
        route = "univariate point forecast"
        series_summary = f"history length={len(series[0])}"

    message_blocks = [
        f"{role.upper()}\n{_wrap_plot_text(content)}" for role, content in messages
    ]
    input_body = (
        f"ROUTE\n{route}\n{series_summary}\nhorizon={horizon!r}\n\n"
        "MESSAGES\n" + "\n\n".join(message_blocks)
    )

    content = result["content"].strip()
    if content:
        content_view = _wrap_plot_text(content)
    elif forecast_values:
        content_view = "(empty; numerical output is plotted below)"
    else:
        content_view = "(empty)"

    if forecast_values:
        raw_values = ", ".join(f"{value:.3f}" for value in forecast_values)
        structured_view = (
            f"raw-scale forecast [{len(forecast_values)}]\n"
            f"{_wrap_plot_text(raw_values)}\n"
            f"normalized forecast [{len(normalized_values)}] available in result"
        )
    else:
        structured_view = "timeseries=None"
    normalization = result["normalization"]
    normalization_view = (
        "normalization=None"
        if normalization is None
        else f"normalization=list[{len(normalization)}]"
    )
    output_body = (
        f"CONTENT\n{content_view}\n\n"
        f"STRUCTURED FIELDS\n{structured_view}\n{normalization_view}\n\n"
        "RUNTIME\n"
        f"finish_reason={result['finish_reason']}  "
        f"prompt_tokens={result['prompt_tokens']}  "
        f"completion_tokens={result['completion_tokens']}\n"
        f"decode_impl={result['decode_impl']}"
    )

    text_lines = max(input_body.count("\n"), output_body.count("\n")) + 1
    text_row_height = max(3.1, 0.23 * text_lines + 0.9)
    plot_height = 2.7
    figure_height = 0.7 + text_row_height + plot_height * len(series)
    fig = plt.figure(figsize=(13, figure_height), constrained_layout=True)
    grid = fig.add_gridspec(
        1 + len(series),
        2,
        height_ratios=[text_row_height] + [plot_height] * len(series),
    )
    _draw_text_panel(
        fig.add_subplot(grid[0, 0]),
        heading="INPUT",
        body=input_body,
        accent=_INPUT_COLOR,
    )
    _draw_text_panel(
        fig.add_subplot(grid[0, 1]),
        heading="MODEL OUTPUT",
        body=output_body,
        accent=_OUTPUT_COLOR,
    )

    for index, (history, name) in enumerate(zip(series, names, strict=True)):
        ax = fig.add_subplot(grid[index + 1, :])
        color = _SERIES_COLORS[index % len(_SERIES_COLORS)]
        history_x = tuple(range(len(history)))
        ax.plot(
            history_x,
            history,
            color=_INPUT_COLOR if horizon is not None else color,
            linewidth=2.0,
            marker="o" if len(history) <= 64 else None,
            markersize=4,
            label="Observed history" if horizon is not None else name,
        )

        if horizon is not None:
            future_x = tuple(range(len(history), len(history) + len(forecast_values)))
            boundary = len(history) - 0.5
            ax.axvspan(boundary, future_x[-1] + 0.5, color=_OUTPUT_COLOR, alpha=0.06)
            ax.plot(
                (len(history) - 1, *future_x),
                (history[-1], *forecast_values),
                color=_OUTPUT_COLOR,
                linewidth=2.0,
                linestyle="--",
                label="Point forecast",
            )
            ax.scatter(future_x, forecast_values, color=_OUTPUT_COLOR, s=24, zorder=3)
            ax.axvline(
                boundary,
                color=_BOUNDARY_COLOR,
                linewidth=1.2,
                linestyle=":",
                label="Forecast boundary",
            )
            ax.text(
                boundary,
                1.01,
                "forecast starts",
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=8,
                color=_BOUNDARY_COLOR,
            )
            plot_title = f"INPUT HISTORY + OUTPUT FORECAST · {name}"
        else:
            plot_title = f"INPUT TIMESERIES · {name}"

        records = result["normalization"]
        if records is not None:
            record = records[index]
            ax.text(
                0.99,
                0.96,
                f"input normalization  μ={record['mean']:.4g}  σ(pop)={record['std']:.4g}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=8,
                color="#555555",
            )
        ax.set_title(plot_title, loc="left", fontsize=11, fontweight="bold")
        _style_series_axis(ax, forecast=horizon is not None)
        ax.legend(loc="upper left", frameon=False, ncol=3)

    fig.suptitle(title, fontsize=15, fontweight="bold")
    return fig


def execute_request(
    *,
    messages: list[dict[str, str]],
    timeseries: list[list[float]] | None = None,
    horizon: int | None = None,
    max_new_tokens: int = 256,
) -> dict:
    model_inputs = processor.apply_chat_template(
        messages,
        timeseries=timeseries,
        horizon=horizon,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            num_return_sequences=1,
        )
    return processor.post_process_generation(outputs, model_inputs=model_inputs)


def run_request(
    *,
    title: str,
    messages: list[dict[str, str]],
    timeseries: list[list[float]] | None = None,
    horizon: int | None = None,
    max_new_tokens: int = 256,
    series_names: Sequence[str] | None = None,
) -> dict:
    message_snapshot = tuple(
        (message["role"], message["content"]) for message in messages
    )
    series_snapshot = tuple(tuple(values) for values in (timeseries or ()))

    result = execute_request(
        messages=messages,
        timeseries=timeseries,
        horizon=horizon,
        max_new_tokens=max_new_tokens,
    )

    fig = render_request_result(
        title=title,
        messages=message_snapshot,
        series=series_snapshot,
        horizon=horizon,
        result=result,
        series_names=series_names,
    )
    plt.show()
    plt.close(fig)
    return result


_CONTRAST_COLORS = ("#D55E00", "#009E73", "#CC79A7")


def run_contrast_requests(
    *,
    title: str,
    series_name: str,
    history: list[float],
    scenarios: list[dict],
    horizon: int | None = None,
    max_new_tokens: int = 256,
) -> list[dict]:
    """Submit the same history under several narratives and overlay the outcomes."""
    results = []
    for scenario in scenarios:
        result = execute_request(
            messages=scenario["messages"],
            timeseries=[list(history)],
            horizon=horizon,
            max_new_tokens=max_new_tokens,
        )
        if horizon is None:
            if result["timeseries"] is not None:
                raise ValueError("A horizon-free request returned numerical output.")
        else:
            values = result["timeseries"]["values"] if result["timeseries"] else ()
            if len(values) != horizon:
                raise ValueError(
                    f"Forecast output length must equal horizon={horizon}, got {len(values)}."
                )
        results.append(result)

    panel_bodies = []
    for scenario, result in zip(scenarios, results, strict=True):
        message_blocks = [
            f"{message['role'].upper()}\n{_wrap_plot_text(message['content'], width=44)}"
            for message in scenario["messages"]
        ]
        if horizon is not None:
            values_text = ", ".join(
                f"{value:.1f}" for value in result["timeseries"]["values"]
            )
            output_view = f"raw-scale forecast [{horizon}]\n{_wrap_plot_text(values_text, width=44)}"
        else:
            output_view = _wrap_plot_text(result["content"].strip(), width=44)
        panel_bodies.append(
            "\n\n".join(message_blocks)
            + f"\n\nMODEL OUTPUT\n{output_view}\n\n"
            + f"finish_reason={result['finish_reason']}  "
            + f"completion_tokens={result['completion_tokens']}"
        )

    text_lines = max(body.count("\n") for body in panel_bodies) + 2
    text_row_height = max(3.1, 0.23 * text_lines + 0.9)
    fig = plt.figure(figsize=(13, text_row_height + 3.6), constrained_layout=True)
    grid = fig.add_gridspec(2, len(scenarios), height_ratios=[text_row_height, 3.2])
    for index, (scenario, body) in enumerate(zip(scenarios, panel_bodies, strict=True)):
        _draw_text_panel(
            fig.add_subplot(grid[0, index]),
            heading=scenario["label"].upper(),
            body=body,
            accent=_CONTRAST_COLORS[index % len(_CONTRAST_COLORS)],
        )

    ax = fig.add_subplot(grid[1, :])
    history_x = tuple(range(len(history)))
    ax.plot(
        history_x,
        history,
        color=_INPUT_COLOR,
        linewidth=2.0,
        marker="o" if len(history) <= 64 else None,
        markersize=4,
        label="Observed history (shared)",
    )
    if horizon is not None:
        boundary = len(history) - 0.5
        future_x = tuple(range(len(history), len(history) + horizon))
        ax.axvspan(boundary, future_x[-1] + 0.5, color="#888888", alpha=0.05)
        ax.axvline(
            boundary,
            color=_BOUNDARY_COLOR,
            linewidth=1.2,
            linestyle=":",
            label="Forecast boundary",
        )
        for index, (scenario, result) in enumerate(
            zip(scenarios, results, strict=True)
        ):
            values = tuple(result["timeseries"]["values"])
            color = _CONTRAST_COLORS[index % len(_CONTRAST_COLORS)]
            ax.plot(
                (len(history) - 1, *future_x),
                (history[-1], *values),
                color=color,
                linewidth=2.0,
                linestyle="--",
                label=scenario["label"],
            )
            ax.scatter(future_x, values, color=color, s=24, zorder=3)
        plot_title = f"SHARED HISTORY + CONTRASTED FORECASTS · {series_name}"
    else:
        plot_title = f"SHARED INPUT TIMESERIES · {series_name}"
    ax.set_title(plot_title, loc="left", fontsize=11, fontweight="bold")
    _style_series_axis(ax, forecast=horizon is not None)
    ax.legend(loc="upper left", frameon=False, ncol=2)
    fig.suptitle(title, fontsize=15, fontweight="bold")
    plt.show()
    plt.close(fig)
    return results

## 1. Text-only completion

This is a compatibility smoke for the language route. Its result has text in `content`, while `timeseries` and `normalization` are `None`. TimeBraid's primary release capabilities are the mixed tasks below.


In [ ]:
text_only_result = run_request(
    title="Text-only completion",
    messages=[
        {
            "role": "user",
            "content": "Explain in one sentence what a moving average reveals about a time series.",
        }
    ],
)

## 2. Single-series description

With `horizon=None`, the series is observed context and the model returns a free-text analysis. The processor reports the normalization record but never emits numerical output on this route.


In [ ]:
description_result = run_request(
    title="Single-series description",
    series_names=["Observed series"],
    messages=[
        {
            "role": "user",
            "content": "Describe the dominant trend, major turning points, and whether the series becomes more volatile.",
        }
    ],
    timeseries=[[2.0, 2.1, 2.2, 2.4, 2.8, 3.1, 3.0, 2.9, 3.4, 3.8, 4.1, 4.3]],
)

## 3. Time-series question answering: anomaly reasoning

Anomaly analysis is also a text task. This example asks for a qualitative location and explanation; the public API does not return anomaly scores or a structured anomaly object.


In [ ]:
anomaly_result = run_request(
    title="Anomaly reasoning",
    series_names=["Observed series"],
    messages=[
        {
            "role": "user",
            "content": "Does this series contain an isolated upward spike? Say only whether it occurs near the beginning, middle, or end; do not give a numeric index. Compare it with the neighboring level.",
        }
    ],
    timeseries=[[20.1, 20.0, 20.2, 20.1, 20.3, 34.8, 20.2, 20.1, 20.0, 20.2]],
)

## 4. Multi-series comparison

Multiple observed series are valid for understanding. Name their meanings in the instruction because the processor appends them as `Series 1`, `Series 2`, and so on. This remains free-text reasoning, not multivariate numerical forecasting.


In [ ]:
comparison_result = run_request(
    title="Multi-series comparison",
    series_names=["Website visits", "Conversion rate"],
    messages=[
        {
            "role": "user",
            "content": "Series 1 is website visits and Series 2 is conversion rate. Compare their trends, volatility, and co-movement.",
        }
    ],
    timeseries=[
        [100.0, 104.0, 107.0, 111.0, 116.0, 120.0, 125.0, 129.0],
        [3.2, 3.1, 3.3, 3.4, 3.8, 3.7, 4.0, 4.2],
    ],
)

## 5. Univariate numerical forecasting

A positive `horizon` selects the point-forecast route. This example uses one history; the processor also accepts multiple histories with an explicit zero-based `target_series_index`. `result["timeseries"]["values"]` contains the forecast on the selected history's original scale; `normalized_values` exposes the z-score-scale model output.


In [ ]:
forecast_result = run_request(
    title="Univariate numerical forecasting",
    series_names=["Seasonal signal"],
    messages=[
        {
            "role": "user",
            "content": "Forecast the next 8 values from the observed seasonal pattern.",
        }
    ],
    timeseries=[
        [
            100.0,
            102.0,
            105.0,
            107.0,
            103.0,
            101.0,
            99.0,
            100.0,
            103.0,
            106.0,
            108.0,
            104.0,
            102.0,
            100.0,
            101.0,
            104.0,
        ]
    ],
    horizon=8,
)

## 6. Text-conditioned numerical forecasting

Forecast instructions may include relevant natural-language context. The structural contract is unchanged: one numerical history, one positive horizon, and one point-forecast sequence.


In [ ]:
contextual_forecast_result = run_request(
    title="Text-conditioned numerical forecasting",
    series_names=["Weekly demand"],
    messages=[
        {
            "role": "system",
            "content": "You analyze weekly product demand.",
        },
        {
            "role": "user",
            "content": "A promotion starts at the first forecast step and is expected to lift demand above the recent seasonal baseline. Forecast the next 6 values.",
        },
    ],
    timeseries=[
        [
            200.0,
            208.0,
            215.0,
            205.0,
            198.0,
            210.0,
            218.0,
            207.0,
            201.0,
            212.0,
            220.0,
            209.0,
        ]
    ],
    horizon=6,
)

## Domain-specific examples

The remaining examples make the language side load-bearing: the same numbers under a different narrative should lead to a different answer. Prompts supply domain context — a symptom report, a clinical variable, an intervention, a service objective — and the response has to combine that context with what the series shows. The data stays synthetic and the API contract is unchanged.

## 7. Patient vitals: symptom attribution

The prompt reports a symptom with an onset day and asks whether the series is consistent with it. Answering requires linking domain knowledge (fever elevates resting heart rate) with the timing and recovery visible in the data, not just describing the shape.

In [ ]:
vitals_result = run_request(
    title="Patient vitals: symptom attribution",
    series_names=["Resting heart rate (bpm)"],
    messages=[
        {
            "role": "user",
            "content": "This series is a patient's daily resting heart rate in beats per minute. The patient reported a fever starting around day 8 that lasted a few days. Is the heart-rate pattern consistent with that report, and does the recovery look complete by the end of the series?",
        }
    ],
    timeseries=[
        [
            63.0,
            62.0,
            64.0,
            63.0,
            65.0,
            64.0,
            63.0,
            72.0,
            88.0,
            91.0,
            86.0,
            74.0,
            66.0,
            64.0,
            63.0,
            62.0,
        ]
    ],
)

## 8. Pulse oximetry: clinical reading

Whether a value is concerning is a domain judgment, not a property of the curve: the same dip would be unremarkable in most other percent-scale series. The model has to apply clinical context to the observed levels and weigh the partial recovery at the end.

In [ ]:
spo2_result = run_request(
    title="Pulse oximetry: clinical reading",
    series_names=["SpO2 (%)"],
    messages=[
        {
            "role": "user",
            "content": "This series is a patient's blood oxygen saturation (SpO2, percent) measured hourly. Are any of these readings clinically concerning, and does the patient appear to be recovering by the end?",
        }
    ],
    timeseries=[
        [
            97.0,
            97.0,
            96.0,
            97.0,
            96.0,
            95.0,
            94.0,
            92.0,
            90.0,
            88.0,
            89.0,
            91.0,
            94.0,
            96.0,
        ]
    ],
)

## 9. Epidemic surveillance: the context changes the forecast

The same history is submitted twice with opposite epidemiological context. Greedy decoding is deterministic, so any difference between the two trajectories is attributable to the text alone. Both forecasts are drawn over the shared history: the school-term context yields a higher crest and a slower decline than the holiday context.

In [ ]:
ili_open_result, ili_closed_result = run_contrast_requests(
    title="Epidemic surveillance: the context changes the forecast",
    series_name="Weekly ILI case count",
    history=[
        420.0,
        380.0,
        395.0,
        410.0,
        430.0,
        485.0,
        560.0,
        660.0,
        790.0,
        940.0,
        1130.0,
        1350.0,
    ],
    horizon=6,
    scenarios=[
        {
            "label": "Schools reopen next week",
            "messages": [
                {
                    "role": "system",
                    "content": "You analyze weekly influenza-like illness case counts for a regional health authority.",
                },
                {
                    "role": "user",
                    "content": "A new school term begins next week, which typically accelerates transmission for several weeks. Forecast the next 6 weekly case counts.",
                },
            ],
        },
        {
            "label": "Schools close next week",
            "messages": [
                {
                    "role": "system",
                    "content": "You analyze weekly influenza-like illness case counts for a regional health authority.",
                },
                {
                    "role": "user",
                    "content": "Schools close for the holidays next week, which typically slows transmission for several weeks. Forecast the next 6 weekly case counts.",
                },
            ],
        },
    ],
)

## 10. Grid load: day-ahead forecast under a heatwave

Two full days of hourly load establish a stable daily shape, and the request asks for one more day. The heatwave sentence is conditioning context, not a guaranteed output property; the plotted forecast should primarily reproduce the daily cycle.

In [ ]:
grid_result = run_request(
    title="Grid load: day-ahead forecast under a heatwave",
    series_names=["Hourly load (GW)"],
    messages=[
        {
            "role": "system",
            "content": "You analyze hourly electricity load for a regional grid operator.",
        },
        {
            "role": "user",
            "content": "The history covers two days of hourly load in gigawatts. A heatwave arrives tomorrow and cooling demand is expected to lift the afternoon and evening peak. Forecast the next 24 hourly values.",
        },
    ],
    timeseries=[
        [
            2.10,
            2.05,
            2.02,
            2.00,
            2.04,
            2.15,
            2.45,
            2.80,
            3.00,
            3.10,
            3.15,
            3.20,
            3.25,
            3.30,
            3.35,
            3.30,
            3.40,
            3.55,
            3.60,
            3.45,
            3.15,
            2.80,
            2.50,
            2.25,
            2.12,
            2.06,
            2.03,
            2.02,
            2.06,
            2.18,
            2.48,
            2.83,
            3.02,
            3.12,
            3.18,
            3.24,
            3.28,
            3.34,
            3.38,
            3.33,
            3.44,
            3.58,
            3.62,
            3.48,
            3.18,
            2.83,
            2.52,
            2.28,
        ]
    ],
    horizon=24,
)

## 11. Service health: latency versus load

Two aligned series support qualitative cross-series reasoning on the understanding route. Beyond describing the relationship, the prompt asks for an operational judgment: whether the behavior needs attention before the next traffic peak.

In [ ]:
service_result = run_request(
    title="Service health: latency versus load",
    series_names=["CPU utilization (%)", "p99 latency (ms)"],
    messages=[
        {
            "role": "user",
            "content": "Series 1 is CPU utilization in percent and Series 2 is p99 request latency in milliseconds for the same service. Describe how latency responds as CPU rises, and say whether this behavior warrants attention before the next traffic peak.",
        }
    ],
    timeseries=[
        [46.0, 48.0, 51.0, 55.0, 58.0, 62.0, 66.0, 70.0, 74.0, 78.0, 82.0, 86.0],
        [
            118.0,
            120.0,
            119.0,
            121.0,
            123.0,
            122.0,
            126.0,
            131.0,
            142.0,
            168.0,
            220.0,
            340.0,
        ],
    ],
)

## 12. Same shape, two readings

Both requests below contain exactly the same numbers, shown once as the shared series. Framed as post-deployment memory usage, the climb is a defect to act on; framed as a patient's step count after knee surgery, it is the desired outcome. The interpretation comes from the narrative, not the curve.

In [ ]:
memory_reading_result, recovery_reading_result = run_contrast_requests(
    title="Same shape, two readings",
    series_name="Shared synthetic series",
    history=[
        42.0,
        45.0,
        44.0,
        48.0,
        47.0,
        51.0,
        50.0,
        54.0,
        53.0,
        57.0,
        56.0,
        60.0,
        59.0,
        63.0,
        62.0,
        66.0,
    ],
    scenarios=[
        {
            "label": "Read as memory usage",
            "messages": [
                {
                    "role": "user",
                    "content": "This series is a service's memory usage in percent, sampled hourly since the latest deployment. Does this pattern suggest a problem the team should act on, and why?",
                }
            ],
        },
        {
            "label": "Read as recovery steps",
            "messages": [
                {
                    "role": "user",
                    "content": "This series is the daily step count, in hundreds of steps, of a patient recovering from knee surgery. Is this pattern a good sign for the recovery, and does the care team need to intervene?",
                }
            ],
        },
    ],
)

## 13. Multiple-choice pattern reading

Time-series question answering is often posed as multiple choice. The format needs no special support: the options are part of the prompt, and the answer is ordinary generated text on the understanding route.

In [ ]:
mcq_result = run_request(
    title="Multiple-choice pattern reading",
    series_names=["Resting heart rate (bpm)"],
    messages=[
        {
            "role": "user",
            "content": "This series is a patient's daily resting heart rate in beats per minute. Which option best describes the pattern? (A) A steady upward trend across the whole series. (B) A sustained mid-series elevation that later resolves back to baseline. (C) A regular periodic oscillation. (D) An abrupt permanent shift to a higher level. Answer with the letter and one sentence of justification.",
        }
    ],
    timeseries=[
        [
            63.0,
            62.0,
            64.0,
            63.0,
            65.0,
            64.0,
            63.0,
            72.0,
            88.0,
            91.0,
            86.0,
            74.0,
            66.0,
            64.0,
            63.0,
            62.0,
        ]
    ],
)

## Result schema and boundaries

Every call returns the same top-level object:

```python
{
    "content": str,
    "timeseries": None | {
        "values": list[float],
        "normalized_values": list[float],
    },
    "normalization": None | list[dict],
    "finish_reason": "stop" | "length",
    "prompt_tokens": int,
    "completion_tokens": int,
    "decode_impl": str,
}
```

Inputs must be finite, non-empty one-dimensional numeric series, and messages must end with a user turn. The public interface is single-request, greedy, one-return-sequence inference from a fresh prompt. It does not expose structured imputation, anomaly scores, class probabilities, embeddings, quantile forecasts, or multivariate numerical forecasts.

Charts show inputs on their raw scales and use `timeseries.values` for the original-scale point forecast. `normalized_values` remains available in the returned result but is not mixed into the raw-scale plot.